# 模型部署与服务化：用 vLLM / SGLang 把 Checkpoint 变成 API

> 前面已经解释了调度、PagedAttention、Prefix Cache、PD 分离等系统机制。这一章不再重复原理。
>
> 目标很明确：**把一个 Hugging Face 模型真正跑成服务。**
>
> 完成后，你应该能：
> - 用 vLLM / SGLang 启动模型
> - 调 OpenAI-compatible API
> - 做流式输出
> - 理解常见启动参数
> - 知道出现 OOM / 吞吐低 / TTFT 高时先看哪里


## 1. 从 Checkpoint 到 Serving Engine

```text
Hugging Face checkpoint
      ↓
Tokenizer / Chat Template
      ↓
vLLM / SGLang / llama.cpp / TensorRT-LLM
      ↓
scheduler + KV cache + kernels
      ↓
OpenAI-compatible HTTP API
      ↓
client / gateway / application
```

Transformers 的 `model.generate()` 很适合开发和验证；真正在线服务通常需要专门的 inference engine。


## 2. vLLM：最小启动

```bash
pip install vllm

vllm serve Qwen/Qwen3-0.6B   --host 0.0.0.0   --port 8000
```

如果机器显存有限，换成更小模型，或根据模型支持情况使用量化 checkpoint。

常见参数里最值得先理解的：

```text
--tensor-parallel-size
--gpu-memory-utilization
--max-model-len
--max-num-seqs
--dtype
--quantization
```


## 3. 用 OpenAI-compatible API 调 vLLM

```python
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
)

resp = client.chat.completions.create(
    model="Qwen/Qwen3-0.6B",
    messages=[{"role": "user", "content": "解释一下 KV Cache"}],
    temperature=0.7,
    max_tokens=128,
)

print(resp.choices[0].message.content)
```

这里你已经能把第 20 章的 sampling 参数和 serving API 对上。


## 4. Streaming：为什么 ChatGPT 看起来是一个字一个字出来？

服务端 Decode 每产生一些 Token 就可以通过 SSE / streaming response 往客户端推。

```python
stream = client.chat.completions.create(
    model="Qwen/Qwen3-0.6B",
    messages=[{"role": "user", "content": "一句话介绍 vLLM"}],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
```


## 5. SGLang：同一个模型怎么启动？

```bash
pip install "sglang[all]"

python -m sglang.launch_server   --model-path Qwen/Qwen3-0.6B   --host 0.0.0.0   --port 30000
```

SGLang 同样可以暴露兼容 API。真正值得比较的不是“哪个命令更短”，而是：

- scheduler 行为
- prefix cache / RadixAttention
- supported quantization
- multi-GPU parallelism
- long-context / chunked prefill
- throughput / TTFT / TPOT


## 6. 一张参数表，把启动参数和原理对上

| 参数 / 配置 | 背后的问题 |
|---|---|
| tensor parallel size | 单卡放不下 / 需要多卡并行 |
| max model len | 最大上下文与 KV Cache 预算 |
| gpu memory utilization | 留多少显存给 KV Cache / runtime |
| max num seqs | 并发 active sequences 上限 |
| quantization | 权重 / KV 低比特路径 |
| prefix caching | 复用重复前缀 |
| chunked prefill | 长 Prompt 调度 |
| speculative config | 减少 Target decode steps |

看到参数时，应该能直接回到 21–24 的系统图。


## 7. OOM、TTFT 高、TPOT 高：怎么第一时间定位？

```text
OOM
 -> 权重太大？
 -> max_model_len 太大？
 -> KV Cache / concurrency 太高？
 -> tensor parallel / quantization 是否合适？

TTFT 高
 -> 排队？
 -> Prompt 太长？
 -> Prefill 被长请求阻塞？
 -> prefix cache / chunked prefill 是否生效？

TPOT 高
 -> decode memory-bound？
 -> batch / concurrency 是否过大？
 -> quantization / kernel / TP 通信开销？
```


## 8. 最小 Benchmark：不要只看“能跑”

最少记录：

```text
model / revision
hardware
dtype / quantization
max context
concurrency
input length / output length
TTFT P50/P95
TPOT P50/P95
throughput tokens/s
peak GPU memory
```

然后再比较 vLLM vs SGLang，或 BF16 vs AWQ / FP8。


## 9. 厂商报告里的 PD 分离，实际部署时怎么理解？

单机 `vllm serve` 是一体化 serving。规模变大以后，系统可能进一步拆成：

```text
Gateway
   ↓
Prefill workers
   ↓  KV transfer
Decode workers
   ↓
Streaming response
```

这就是前一章的 **PD 分离** 落到部署拓扑后的样子。

所以读厂商报告时，先问：
1. Prefill 和 Decode 是否分池？
2. KV 怎么传？
3. scheduler 在哪里？
4. 目标是 TTFT、TPOT、吞吐还是成本？


## 10. 到这里，你应该能读懂一条典型招聘 JD

例如：

> 熟悉 vLLM / SGLang，理解 PagedAttention、Continuous Batching、Prefix Caching、Chunked Prefill、Speculative Decoding、PD Disaggregation；有量化和多卡推理经验。

现在这些词已经可以落回完整链路：

```text
Sampling
→ Prefill / Decode
→ KV Cache
→ Quantization
→ Speculative Decoding
→ Scheduler
→ PagedAttention
→ Prefix Cache / RadixAttention
→ Chunked Prefill
→ PD Disaggregation
→ vLLM / SGLang deployment
→ Benchmark
```

这就是 Part 3 的最终目标：**不仅“听过名词”，而是知道它为什么出现、在哪一层、怎么跑起来。**
